# Fine-tune Didi (llama3.2:1b)

Runtime > Change runtime type > T4 GPU.

Upload `train.jsonl` and `test.jsonl` to the Colab Files panel before running the cells below.

In [1]:
# 1. Install (if this fails, copy the install cell from Unsloth's current Colab notebook)
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 46.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 53.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.4 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0━━━  0/13 [torchao]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━  6/13 [xformers]er]
    Found existing installation: transformers 5.22.1━━━━━━━━━━  6/13 [xformers]
    Uninstalling transformers-5.22.1:

In [2]:
# 2. Settings. The base model must match the Ollama baseline (llama3.2:1b).
# Check the model licence first before any use beyond this demo.
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
SYSTEM = (
    "You are Didi, a maternal and child health assistant for ASHA workers "
    "in rural India. You give accurate, safe health information about pregnancy, "
    "antenatal care, nutrition, and newborn care. "
    "You always respond in English. "
    "You never give dangerous medical advice. "
    "You always refer critical cases to a doctor or hospital immediately."
)
MAX_SEQ = 2048

In [3]:
# 3. Load the model in 4 bit and add LoRA adapters
from unsloth import FastLanguageModel

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1113 06:07:05.279000 1179 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1113 06:07:05.318000 1179 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.11.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


In [4]:
# 4. Load train.jsonl and turn each chat into one training text
from datasets import load_dataset

ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = ds.map(lambda ex: {"text": tok.apply_chat_template(
    ex["messages"], tokenize=False, add_generation_prompt=False)})
print(len(ds), "examples")
print(ds[0]["text"][:600])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

67 examples
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 13 Nov 2026

You are Didi, a maternal and child health assistant for ASHA workers in rural India. You give accurate, safe health information about pregnancy, antenatal care, nutrition, and newborn care. You always respond in English. You never give dangerous medical advice. You always refer critical cases to a doctor or hospital immediately.<|eot_id|><|start_header_id|>user<|end_header_id|>

How can I check if a woman is pregnant?<|eot_id|><|start_header_id|>assistant<|end_header_id|>


In [ ]:
# 5. Train. If an argument name errors, copy the trainer cell from Unsloth's
# current notebook and keep the dataset cell above.
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=6,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="outputs",
        report_to="none",
        seed=3407,
    ),
)
trainer.train()

In [6]:
# 6. Save the adapter to Drive so a session reset does not lose it
from google.colab import drive

drive.mount("/content/drive")
model.save_pretrained("/content/drive/MyDrive/didi-lora")
tok.save_pretrained("/content/drive/MyDrive/didi-lora")

Mounted at /content/drive


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/didi-lora/tokenizer_config.json.


('/content/drive/MyDrive/didi-lora/tokenizer_config.json',
 '/content/drive/MyDrive/didi-lora/chat_template.jinja',
 '/content/drive/MyDrive/didi-lora/tokenizer.json')

In [7]:
# 7. Answer the test prompts here and save answers_tuned.csv (the safe fallback)
import csv
import json

FastLanguageModel.for_inference(model)
rows = []
for line in open("test.jsonl", encoding="utf-8"):
    case = json.loads(line)
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": case["prompt"]}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=400, temperature=0.2, do_sample=True)
    answer = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    rows.append({"id": case["id"], "answer": answer.strip()})

with open("answers_tuned.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "answer"])
    writer.writeheader()
    writer.writerows(rows)
print("Saved answers_tuned.csv. Download it from the Files panel.")

Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tok

Saved answers_tuned.csv. Download it from the Files panel.


In [8]:
# 8. Export for Ollama. Look for the .gguf file in the folder it prints, then download it.
model.save_pretrained_gguf("didi-ft", tok, quantization_method="q4_k_m")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in didi-ft/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Unsloth: Merge process complete. Saved to `/content/didi-ft`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12319-mix-58871cd (app-b12319-mix-58871cd-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['didi-ft_gguf/Llama-3.2-1B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['didi-ft_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model didi-ft_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to didi-ft_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f didi-ft_gguf/Modelfile


{'save_directory': 'didi-ft',
 'gguf_directory': 'didi-ft_gguf',
 'gguf_files': ['didi-ft_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'didi-ft_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': True}